# Project 08: Viterbi Algorithm for Hidden Markov Models

Viterbi Algorithm

Phase A: Build the model (once per model)
Step 1: Store the model. Save the initial, transition, and emission dictionaries exactly as the user gives them
Step 2: Derive the states and the alphabet. States = keys of init_probs, in a fixed order, each with a row number (I -> 0, G -> 1). Alphabet = symbols in the emission rows ({A, C, G, T})
Step 3: Check the model. Same states in every dictionary, same symbols in every emission row, no negative probabilities, and every row adds up to 1 (within a small tolerance). Checked on the raw numbers, because log values no longer add up to 1
Step 4: Convert to log space. Take the log of every probability once, so multiplying becomes adding (Durbin p. 77-78)

Phase B: Decode (once per observation sequence)
Step 5: Check the sequence. Not empty, and every symbol is in the alphabet ("GG" is fine; "GGN" is not)
Step 6: Build the empty grids. Score grid filled with -inf, traceback grid filled with -1. Rows = states, columns = positions
Step 7: First column. For each state: log initial + log emission of the first symbol. No max and no pointer, because nothing comes before position 1
Step 8: Every other column, left to right. For each current state, compare every previous state: (its score one column left + log transition to the current state). Keep the biggest, remember which previous state won, then add the log emission of this column's symbol
Step 9: Finish. In the last column only, the biggest score = the best path's log-probability, and its row = the best final state
Step 10: Trace back. Start at the best final state, follow the pointers left to column 0, reverse the list, and turn row numbers back into state names

Functions in project below (which Steps they match, and what they do)

- `HMM.__init__(init_probs, trans_probs, emit_probs)`: Steps 1, 2, 4. Stores the model, derives the states and alphabet, converts to logs
- `validate()`: Step 3. Checks the raw dictionaries; raises ValueError if broken
- `check_observations(observations)`: Step 5. Checks the sequence; raises ValueError if broken
- `initialize_matrix(num_states, num_positions)`: Step 6. Builds the empty score and traceback grids
- `update_matrix(row, column, score, previous_row)`: Steps 7-8, one box. Writes one score and its pointer
- `traceback(best_final_row)`: Step 10. Follows the pointers back and returns the path
- `viterbi_algorithm(observations)`: Steps 5-10, the whole decode. Runs the full decode and returns the path and log-probability

Viterbi Generic Pseudocode

Build The Model
    states = keys of init_probs, in order; each gets a row number
    alphabet = symbols in the emission rows
    check: same states everywhere, same symbols in every emission row, no negatives, every row adds to 1
    log_init, log_trans, log_emit = log of every probability

Decode
    check the sequence: not empty, every symbol in the alphabet
    score grid = states x positions, all -inf;  traceback grid = same size, all -1

    first column, for each state:
        score = log_init[state] + log_emit[state][first symbol]

    for each column after the first:
        for each current state:
            best = biggest over previous states of (score[previous, column - 1] + log_trans[previous][current])
            score[current, column] = best + log_emit[current][this column's symbol]
            pointer[current, column] = the previous state that gave best

    best final state = row with the biggest score in the last column
    path = follow pointers back from the best final state, then reverse
    return path, biggest score

In [33]:
from typing import Dict, List, Set, Tuple

import numpy as np

In [34]:
# Example data structures from the Project 08 instructions

obs = "GGCACTGAA"

init_probs = {
    "I": 0.2,
    "G": 0.8
}

trans_probs = {
    "I": {"I": 0.7, "G": 0.3},
    "G": {"I": 0.1, "G": 0.9}
}

emit_probs = {
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "G": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3}
}

In [35]:
class HMM:
    """Hidden Markov model built from initial, transition, and emission probabilities.

    Attributes:
        init_probs (Dict[str, float]): initial probabilities as given, {state: probability}
        trans_probs (Dict[str, Dict[str, float]]): transitions as given, {from_state: {to_state: probability}}
        emit_probs (Dict[str, Dict[str, float]]): emissions as given, {state: {symbol: probability}}
        states (List[str]): hidden states in a fixed order, i.e. ['I', 'G']
        state_index (Dict[str, int]): each state's row number, i.e. {'I': 0, 'G': 1}
        alphabet (Set[str]): observable symbols, i.e. {'A', 'C', 'G', 'T'}
        log_init (Dict[str, float]): init_probs with every probability as a log
        log_trans (Dict[str, Dict[str, float]]): trans_probs with every probability as a log
        log_emit (Dict[str, Dict[str, float]]): emit_probs with every probability as a log
        score_matrix (np.ndarray): best-path score for each state at each position
        traceback_matrix (np.ndarray): row number of the winning previous state for each box
    """

    def __init__(
        self,
        init_probs: Dict[str, float],
        trans_probs: Dict[str, Dict[str, float]],
        emit_probs: Dict[str, Dict[str, float]]
    ) -> None:
        """Stores the model, gets the states and alphabet, checks the inputs, and converts to logs.

        Args:
            init_probs (Dict[str, float]): chance of starting in each state
            trans_probs (Dict[str, Dict[str, float]]): chance of moving from each state to each state
            emit_probs (Dict[str, Dict[str, float]]): chance of each state emitting each symbol

        Raises:
            ValueError: if the dictionaries fail any check in validate()

        Examples:
            >>> hmm = HMM(init_probs, trans_probs, emit_probs)
            >>> hmm.states
            ['I', 'G']
            >>> hmm.state_index
            {'I': 0, 'G': 1}
        """
        #Step 1. Store the model
        self.init_probs = init_probs  #the initial probabilities exactly as the user gave them
        self.trans_probs = trans_probs  #the transition probabilities exactly as the user gave them
        self.emit_probs = emit_probs  #the emission probabilities exactly as the user gave them

        #Step 2. Get the states and the alphabet from the dictionaries
        self.states = list(self.init_probs.keys())  #the state names from init_probs, kept in that order, i.e. ['I', 'G']
        self.state_index = {}  #empty dictionary; will hold each state's row number in the grids
        for row, state in enumerate(self.states):  #go through the states, counting rows from 0#
            self.state_index[state] = row  #'I' gets row 0, 'G' gets row 1
        first_state = self.states[0]  #the first state in the list, i.e. 'I'
        self.alphabet = set(self.emit_probs[first_state].keys())  #the symbols that state can emit, i.e. {'A', 'C', 'G', 'T'}

        #Step 3. Check the model
        self.validate()  #runs all the model checks on the original probabilities; stops with an error if anything is wrong

        #Step 4. Convert every probability to its log
        self.log_init = {}  #empty dictionary; will hold the log of each initial probability
        for state, probability in self.init_probs.items():  #go through each state and its probability
            self.log_init[state] = np.log(probability)  #log of 0.2 = -1.6094

        self.log_trans = {}  #empty dictionary; will hold the log of each transition probability
        for from_state, row in self.trans_probs.items():  #go through each state's row of transitions
            self.log_trans[from_state] = {}  #empty dictionary for this row
            for to_state, probability in row.items():  #go through each destination state in the row
                self.log_trans[from_state][to_state] = np.log(probability)  #log of 0.9 = -0.1054

        self.log_emit = {}  #empty dictionary; will hold the log of each emission probability
        for state, row in self.emit_probs.items():  #go through each state's row of emissions
            self.log_emit[state] = {}  #empty dictionary for this row
            for symbol, probability in row.items():  #go through each symbol in the row
                self.log_emit[state][symbol] = np.log(probability)  #log of 0.4 = -0.9163

        self.score_matrix = None  #score_matrix will hold the best-path score for each state at each position, later in initialize_matrix() and viterbi_algorithm()
        self.traceback_matrix = None  #traceback_matrix will hold the row number of the winning previous state for each box, later in initialize_matrix() and viterbi_algorithm()


    def validate(self) -> None:
        """Checks that the model's dictionaries match each other and are real probabilities.

        Raises:
            ValueError: with a message naming the check that failed
        """
        state_set = set(self.states)  #the states as a set so they can be compared, i.e. {'I', 'G'}

        #Same states everywhere
        if set(self.trans_probs.keys()) != state_set:  #the states in trans_probs don't match the states in init_probs
            raise ValueError("transition states don't match the initial states")  #stops with an error
        for from_state, row in self.trans_probs.items():  #go through each state's row of transitions
            if set(row.keys()) != state_set:  #this row's destination states don't match
                raise ValueError(f"transition row {from_state} has the wrong states")  #stops with an error
        if set(self.emit_probs.keys()) != state_set:  #the states in emit_probs don't match the states in init_probs
            raise ValueError("emission states don't match the initial states")  #stops with an error

        #Same symbols in every emission row
        for state, row in self.emit_probs.items():  #go through each state's row of emissions
            if set(row.keys()) != self.alphabet:  #this row's symbols don't match the alphabet
                raise ValueError(f"emission row {state} has different symbols")  #stops with an error

        #No negative probabilities
        for probability in self.init_probs.values():  #go through every initial probability
            if probability < 0:
                raise ValueError("initial probabilities can't be negative")  #stops with an error
        for row in self.trans_probs.values():  #go through every transition row
            for probability in row.values():  #go through every probability in the row
                if probability < 0:
                    raise ValueError("transition probabilities can't be negative")  #stops with an error
        for row in self.emit_probs.values():  #go through every emission row
            for probability in row.values():  #go through every probability in the row
                if probability < 0:
                    raise ValueError("emission probabilities can't be negative")  #stops with an error

        #Everything adds up to 1
        if not np.isclose(sum(self.init_probs.values()), 1):  #the initial probabilities don't add up to 1; isclose allows a tiny rounding difference
            raise ValueError("initial probabilities don't add up to 1")
        for from_state, row in self.trans_probs.items():  #go through each transition row
            if not np.isclose(sum(row.values()), 1):  #this row doesn't add up to 1
                raise ValueError(f"transition row {from_state} doesn't add up to 1")  #stops with an error
        for state, row in self.emit_probs.items():  #go through each emission row
            if not np.isclose(sum(row.values()), 1):  #this row doesn't add up to 1
                raise ValueError(f"emission row {state} doesn't add up to 1")  #stops with an error


    def check_observations(self, observations: str) -> None:
        """Checks that an observation sequence can be decoded by this model.

        Args:
            observations (str): observed symbols, i.e. "GGCACTGAA"

        Raises:
            ValueError: if observations is empty, or has a symbol not in the alphabet
        """
        if len(observations) == 0:  #the sequence is empty
            raise ValueError("observation sequence is empty")  #stops with an error

        unknown_symbols = set(observations) - self.alphabet  #the symbols in observations that are not in the alphabet, i.e. {'N'}
        if unknown_symbols:  #there is at least one unknown symbol
            raise ValueError(f"symbols not in the alphabet: {unknown_symbols}")  #stops with an error


    def initialize_matrix(self, num_states: int, num_positions: int) -> None:
        """Builds the empty score and traceback grids for one decode.

        Args:
            num_states (int): number of rows, one per hidden state
            num_positions (int): number of columns, one per observation

        Returns:
            None: sets score_matrix and traceback_matrix
        """
        self.score_matrix = np.full((num_states, num_positions), -np.inf)  #a grid with num_states rows and num_positions columns, every box set to negative infinity, meaning not filled in yet
        self.traceback_matrix = np.full((num_states, num_positions), -1, dtype=int)  #a grid the same size for whole numbers, every box set to -1, meaning "no previous state"


    def update_matrix(self, row: int, column: int, score: float, previous_row: int) -> None:
        """Writes one box: its score and the row of the previous state that won.

        Args:
            row (int): row number of the current state
            column (int): position in the observation sequence
            score (float): best-path score for this box
            previous_row (int): row number of the winning previous state (-1 in the first column)

        Returns:
            None: changes score_matrix and traceback_matrix
        """
        self.score_matrix[row, column] = score  #write the score in this box
        self.traceback_matrix[row, column] = previous_row  #write the row of the previous state that won in this box


    def traceback(self, best_final_row: int) -> List[str]:
        """Follows the pointers back from the best final state to build the path.

        Args:
            best_final_row (int): row number of the best state in the last column

        Returns:
            List[str]: hidden states from the first position to the last
        """
        path_rows = [best_final_row]  #a list that starts with the best final state's row
        last_column = self.traceback_matrix.shape[1] - 1  #shape[1] is the number of columns; minus 1 because counting starts at 0

        for column in range(last_column, 0, -1):  #go from the last column back down to column 1; column 0 has no pointer
            current_row = path_rows[-1]  #the last row added to path_rows
            previous_row = self.traceback_matrix[current_row, column]  #the pointer stored in this box
            path_rows.append(previous_row)  #add it to the end of path_rows

        path_rows.reverse()  #flip the list so it runs from the first position to the last

        path = []  #empty list; will hold the state names
        for row in path_rows:  #go through each row number
            path.append(self.states[row])  #turn the row number into its state name, i.e. 1 becomes 'G'

        return path


    def viterbi_algorithm(self, observations: str) -> Tuple[List[str], float]:
        """Finds the single most probable sequence of hidden states.

        Args:
            observations (str): observed symbols, i.e. "GGCACTGAA"

        Returns:
            Tuple[List[str], float]: the best path (one state per observation)
                and its log-probability

        Examples:
            >>> hmm.viterbi_algorithm("G")
            (['G'], -1.8326)
            >>> hmm.viterbi_algorithm("GG")
            (['G', 'G'], -3.5474)
            >>> hmm.viterbi_algorithm("GGCACTGAA")
            (['G', 'G', 'G', 'G', 'G', 'G', 'G', 'G', 'G'], -13.9291)
        """
        #SETUP
        self.check_observations(observations)  #stops with an error if the sequence is empty or has an unknown symbol
        num_states = len(self.states)  #how many states there are, i.e. 2
        num_positions = len(observations)  #how many symbols are in observations, i.e. 9
        self.initialize_matrix(num_states, num_positions)  #builds the empty grids

        #Step 7. First column (no max, no pointer)
        first_symbol = observations[0]  #the first symbol, i.e. 'G'
        for state in self.states:  #go through each state)
            score = self.log_init[state] + self.log_emit[state][first_symbol]  #log of starting in this state + log of this state emitting the first symbol
            self.update_matrix(self.state_index[state], 0, score, -1)  #write the score in this state's row, column 0, with -1 as the pointer

        #Step 8. Fill the rest of the columns, left to right
        for column in range(1, num_positions):  #go through column 1 to the last column
            symbol = observations[column]  #the symbol at this column
            for current_state in self.states:  #go through each state for this column
                best_candidate = -np.inf  #start at negative infinity, so any real score beats it
                best_previous = self.states[0]  #start with the first state
                for previous_state in self.states:  #compare every state in the column to the left
                    candidate = self.score_matrix[self.state_index[previous_state], column - 1] + self.log_trans[previous_state][current_state]  #previous state's score + log of moving from previous_state to current_state
                    if candidate > best_candidate:  #this way in is better than the best so far
                        best_candidate = candidate  #keep the better score
                        best_previous = previous_state  #remember which state it came from
                score = best_candidate + self.log_emit[current_state][symbol]  #best way in + log of current_state emitting this symbol
                self.update_matrix(self.state_index[current_state], column, score, self.state_index[best_previous])  #write the score, with the winning previous state's row as the pointer

        #Step 9. Finish
        last_column = num_positions - 1  #the number of the last column
        best_final_row = int(np.argmax(self.score_matrix[:, last_column]))  #the row with the biggest score in the last column; [:, last_column] means every row of that column
        best_log_prob = self.score_matrix[best_final_row, last_column]  #that biggest score

        #Step 10. Trace back
        path = self.traceback(best_final_row)  #follows the pointers back from the best final state

        return path, best_log_prob

In [36]:
#Test cell for __init__

hmm = HMM(init_probs, trans_probs, emit_probs)  #build the model from the example data

#Test cases:
    #hmm.states                    ['I', 'G']
    #hmm.state_index               {'I': 0, 'G': 1}
    #hmm.alphabet                  {'A', 'C', 'G', 'T'}
    #hmm.log_init['I']             about -1.6094  (log of 0.2)
    #hmm.log_trans['G']['G']       about -0.1054  (log of 0.9)
    #hmm.log_emit['I']['C']        about -0.9163  (log of 0.4)
  
print("states:", hmm.states, "expected: ['I', 'G']")
print("state_index:", hmm.state_index, "expected: {'I': 0, 'G': 1}")
print("alphabet:", hmm.alphabet, "expected: {'A', 'C', 'G', 'T'}")
print("log_init I:", round(hmm.log_init['I'], 4), "expected: -1.6094")
print("log_trans G to G:", round(hmm.log_trans['G']['G'], 4), "expected: -0.1054")
print("log_emit I, C:", round(hmm.log_emit['I']['C'], 4), "expected: -0.9163")

states: ['I', 'G'] expected: ['I', 'G']
state_index: {'I': 0, 'G': 1} expected: {'I': 0, 'G': 1}
alphabet: {'C', 'A', 'G', 'T'} expected: {'A', 'C', 'G', 'T'}
log_init I: -1.6094 expected: -1.6094
log_trans G to G: -0.1054 expected: -0.1054
log_emit I, C: -0.9163 expected: -0.9163


In [37]:
#Test cell for validate() each bad model should stop with an error

#Test cases (bad input, expected error):
    #bad_init = {'I': 0.5, 'G': 0.4}                                                   initial probabilities don't add up to 1
    #bad_trans = {'I': {'I': 0.7, 'G': 0.2}, 'G': {'I': 0.1, 'G': 0.9}}                transition row I doesn't add up to 1
    #bad_emit_states = {'I': emit_probs['I']}                                          emission states don't match (G row missing)
    #bad_emit_symbols = {'I': {'A': 0.1, 'C': 0.4, 'G': 0.5}, 'G': emit_probs['G']}    emission row G has different symbols (the alphabet comes from the I row, which is missing T, so G is the row that doesn't match)
    #bad_negative = {'I': -0.2, 'G': 1.2}                                              initial probabilities can't be negative

bad_init = {'I': 0.5, 'G': 0.4}
bad_trans = {'I': {'I': 0.7, 'G': 0.2}, 'G': {'I': 0.1, 'G': 0.9}}
bad_emit_states = {'I': emit_probs['I']}
bad_emit_symbols = {'I': {'A': 0.1, 'C': 0.4, 'G': 0.5}, 'G': emit_probs['G']}
bad_negative = {'I': -0.2, 'G': 1.2}

bad_models = [  #each item is (initial, transition, emission), with one bad dictionary in each
    (bad_init, trans_probs, emit_probs),
    (init_probs, bad_trans, emit_probs),
    (init_probs, trans_probs, bad_emit_states),
    (init_probs, trans_probs, bad_emit_symbols),
    (bad_negative, trans_probs, emit_probs),
]

#FOR each bad input:
for test_init, test_trans, test_emit in bad_models:  #unpack the three dictionaries for this test
    try:  #try to build the model; if it stops with an error, jump to except
        HMM(test_init, test_trans, test_emit)   #build the model with the bad input in place of the good one)
        print("no error")  #only prints if the check failed to catch the problem
    except ValueError as error:  #catch the error so the notebook keeps running
        print(error)  #print the error message to compare with the expected error

initial probabilities don't add up to 1
transition row I doesn't add up to 1
emission states don't match the initial states
emission row G has different symbols
initial probabilities can't be negative


In [38]:
#Test cell for check_observations()

hmm = HMM(init_probs, trans_probs, emit_probs)  #build the model from the example data

#Test cases (input, expected answer):
    #"GG"            passes (does not need every symbol)
    #"GGCACTGAA"     passes
    #"GGNA"          error that names 'N'
    #""              error: empty sequence

for test_input in ["GG", "GGCACTGAA", "GGNA", ""]:  #go through each test case
   
    try:  #try the check; if it stops with an error, jump to except
        hmm.check_observations(test_input)
        print(test_input, "passed")
    except ValueError as error:  #catch the error so the notebook keeps running
        print(test_input, "error:", error)

GG passed
GGCACTGAA passed
GGNA error: symbols not in the alphabet: {'N'}
 error: observation sequence is empty


In [39]:
#Test cell for initialize_matrix() and update_matrix()

hmm = HMM(init_probs, trans_probs, emit_probs)  #build the model from the example data
hmm.initialize_matrix(2, 3)  #2 states, 3 positions
print(hmm.score_matrix)  #expected: 2 rows x 3 columns, every box -inf
print(hmm.traceback_matrix)  #expected: 2 rows x 3 columns, every box -1

hmm.update_matrix(1, 0, -1.8326, -1)  #row 1, column 0, score -1.8326, previous row -1
print(hmm.score_matrix[1, 0])  #expected: -1.8326
print(hmm.traceback_matrix[1, 0])  #expected: -1

[[-inf -inf -inf]
 [-inf -inf -inf]]
[[-1 -1 -1]
 [-1 -1 -1]]
-1.8326
-1


In [40]:
#Test cell for traceback()

hmm = HMM(init_probs, trans_probs, emit_probs)  #build the model from the example data
hmm.viterbi_algorithm("GG")  #fills the grids

print(hmm.traceback(1))  #start from row 1, G; expected: ['G', 'G']
print(hmm.traceback(0))  #pstart from row 0, I; expected: ['I', 'I'], because the I box in column 2 points back to I

['G', 'G']
['I', 'I']


In [41]:
#Test cell for viterbi_algorithm()

hmm = HMM(init_probs, trans_probs, emit_probs)  #build the model from the example data

#Test cases (input, expected path, expected log-probability):
    #"G"              ['G']                  about -1.8326   (log of 0.8 x 0.2 = 0.16)
    #"GG"             ['G', 'G']             about -3.5474   (log of 0.0288, worked by hand)
    #"GGCACTGAA"      ['G'] nine times       about -13.9291


for test_input in ["G", "GG", "GGCACTGAA"]:  #go through each test case
    path, log_prob = hmm.viterbi_algorithm(test_input)  #run Viterbi; it returns the path and the log-probability
    print(test_input, path, round(log_prob, 4))  #round to 4 decimals to compare with the expected number

#Underflow test
long_obs = "GGCACTGAA" * 200  #the example sequence repeated 200 times, 1800 symbols
path, log_prob = hmm.viterbi_algorithm(long_obs)
print(log_prob)  #expected: a real negative number, not 0, -inf, or nan
print(len(path))  #expected: 1800

G ['G'] -1.8326
GG ['G', 'G'] -3.5474
GGCACTGAA ['G', 'G', 'G', 'G', 'G', 'G', 'G', 'G', 'G'] -13.9291
-2762.3828671147444
1800


In [42]:
#Test cell for a model with 3 states and a non-DNA alphabet (nothing hard-coded for 2 states or for DNA)


init_3 = {'X': 0.5, 'Y': 0.3, 'Z': 0.2}  #initial probabilities for 3 states

trans_3 = {'X': {'X': 0.6, 'Y': 0.2, 'Z': 0.2},  #transition probabilities; each row adds up to 1
           'Y': {'X': 0.3, 'Y': 0.4, 'Z': 0.3},
           'Z': {'X': 0.1, 'Y': 0.1, 'Z': 0.8}}

emit_3 = {'X': {'H': 0.9, 'T': 0.1},  #emission probabilities for a heads/tails alphabet
          'Y': {'H': 0.5, 'T': 0.5},
          'Z': {'H': 0.2, 'T': 0.8}}

hmm_3 = HMM(init_3, trans_3, emit_3)   #build the 3-state model

#Test case "T": by hand X = 0.5 x 0.1 = 0.05, Y = 0.3 x 0.5 = 0.15, Z = 0.2 x 0.8 = 0.16, so Z wins
path, log_prob = hmm_3.viterbi_algorithm("T")
print(path, round(log_prob, 4))  #expected: ['Z'] -1.8326, the log of 0.16

['Z'] -1.8326


In [43]:
if __name__ == "__main__":
    
    hmm = HMM(init_probs, trans_probs, emit_probs)  #build the model from the example data
    path, log_prob = hmm.viterbi_algorithm(obs)  #decode the example sequence
    print(obs)  #the observation sequence
    print("".join(path))     #join the list of states into one string, i.e. GGGGGGGGG
    print(log_prob)      #the best path's log-probability

GGCACTGAA
GGGGGGGGG
-13.929108456051065
